# Studio / 1BR rent model — Santa Cruz & Monterey

Predicting asking rent for studios and one-bedrooms from listing characteristics.
Data: 3,038 listing events, 2020–2026, from `data/processed/features.parquet`.

| model | MAE | MAPE | note |
|---|---|---|---|
| baseline — median by place × bedrooms | $367 | 15.2% | the bar to beat |
| ridge | $316 | 14.0% | + a 23-feature variant for readable coefficients |
| LightGBM | $292 | 12.4% | cannot extrapolate past training dates |
| **LightGBM + trend** | **$295** | **12.6%** | **the model** — works on future dates |
| + calibrated intervals | | | 81% coverage, ×/÷ 1.24 |

Run top to bottom. Restart the kernel first if you have been editing.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, LinearRegression

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 60)

df = pd.read_parquet("../data/processed/features.parquet")

# the modelling slice: studios and 1-bedrooms in the two target counties
t = df[df.is_studio_1br & df.county.isin(["Santa Cruz", "Monterey"])].copy()

print(f"all events   {len(df):,}")
print(f"studio/1br   {len(t):,}   across {t.id.nunique():,} properties")
print(f"date range   {t.listedDate.min():%Y-%m} .. {t.listedDate.max():%Y-%m}")

## 1. The target

Rent is right-skewed, so we model `log_price` and convert back with `np.exp()`
at the end. Rents rose ~29% over the window, which the split has to respect.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
t.price.hist(bins=50, ax=ax[0]);    ax[0].set_title("price — skewed")
t.log_price.hist(bins=50, ax=ax[1]); ax[1].set_title("log_price — usable")
plt.tight_layout(); plt.show()

print(t.groupby(t.listedDate.dt.year).price.agg(["size", "median"]).to_string())

## 2. Train / test split

Two separate leakage traps, and they conflict with each other.

**Time.** Rents rose ~29% across the window. A random split lets the model see
the future. So we split on a date: train on older, test on newer.

**Repeated properties.** `id` is address-derived and the same unit recurs across
years. Worse, `latitude`/`longitude` effectively fingerprint a building — a tree
can split finely enough to memorise "the unit at this coordinate rents for $X".
So no *building* may appear on both sides. We group on rounded coordinates
rather than `id`, which also separates `Apt 1` from `Apt 2` in the same complex.

**The conflict.** A property listed in 2021 *and* 2026 belongs in train by date
and in test by date. We drop those from **train**, protecting the test set:
losing 11% of train costs a little accuracy, a compromised test set costs the
ability to measure anything.

Cutoff of 2026-01-01 comes from `t.listedDate.quantile(0.75)`, which lands on
2026-01-08 — rounded to a clean month boundary.

In [ ]:
CUTOFF = pd.Timestamp("2026-01-01")

t["coord"] = t.latitude.round(5).astype(str) + "," + t.longitude.round(5).astype(str)

train = t[t.listedDate <  CUTOFF]
test  = t[t.listedDate >= CUTOFF].copy()

overlap = set(train.coord) & set(test.coord)
train = train[~train.coord.isin(overlap)].copy()

def describe_split(train, test):
    for name, d in [("train", train), ("test", test)]:
        print(f"{name:<6} events {len(d):>6,}  properties {d.id.nunique():>6,}  "
              f"median ${d.price.median():>6,.0f}  "
              f"{d.listedDate.min():%Y-%m} .. {d.listedDate.max():%Y-%m}")

describe_split(train, test)
print(f"\nbuildings dropped from train: {len(overlap)}")
print(f"remaining overlap: {len(set(train.coord) & set(test.coord))}  (must be 0)")

Train's median ($2,100) sits well below test's ($2,295), but that is not a
bug — train spans six years of rising rents while test is 2026 only. Compare
like with like: 2025 train is $2,250 against 2026 test at $2,295, one year of
normal drift.

In [ ]:
print(train.groupby(train.listedDate.dt.year).price.agg(["size", "median"]).to_string())

## 3. Metrics

- **MAE** — average dollars off. What you say out loud.
- **MAPE** — average % off. Comparable across price levels and neighbourhoods.
- **bias** — average *signed* error. Near zero means errors cancel; consistently
  negative means the model systematically under-predicts.

In [ ]:
results = []

def evaluate(pred, label, show=True):
    pred = pd.Series(np.asarray(pred), index=test.index)
    e = pred - test["price"]
    row = {"model": label, "MAE": e.abs().mean(),
           "MAPE": (e.abs() / test["price"]).mean() * 100, "bias": e.mean()}
    results.append(row)
    if show:
        print(f"{label:<26} MAE ${row['MAE']:>5,.0f}   "
              f"MAPE {row['MAPE']:>4.1f}%   bias ${row['bias']:>+6,.0f}")
    return pred

## 4. Baseline — median by place × bedrooms

A lookup table, no learning. Any model that cannot beat this is not worth its
complexity. 10 test rows hit a `place` × `bedrooms` combination absent from
train; they fall back to the median for their bedroom count, which keeps the
strongest signal (unit size) when location cannot be resolved.

In [ ]:
lookup = train.groupby(["place", "bedrooms"])["price"].median()
lookup_df = lookup.reset_index().rename(columns={"price": "baseline"})
test = test.merge(lookup_df, on=["place", "bedrooms"], how="left")

fallback = test.bedrooms.map({0: train[train.bedrooms == 0].price.median(),
                              1: train[train.bedrooms == 1].price.median()})
test["baseline"] = test["baseline"].fillna(fallback)

print(f"{len(lookup)} groups; {int((lookup.groupby(level=0).size() >= 0).sum())} places")
print(f"groups backed by <5 listings: {(train.groupby(['place','bedrooms']).size() < 5).sum()}"
      f" of {len(lookup)}   <- baseline is fragile in thin neighbourhoods\n")
evaluate(test["baseline"], "baseline");

The baseline gives **every** 1BR in Santa Cruz the same $2,395, while actual
rents there run $1,095–$4,518. It cannot vary within a place at all — that
missing variation is what the models below have to explain.

## 5. Ridge

Needs three preprocessing steps LightGBM will not: fill missing values, encode
categoricals as numbers, scale to comparable ranges. All three are learned from
**train only** and applied to test.

One-hot columns are deliberately **not** scaled. Scaling divides by standard
deviation, and for a category with one listing that multiplies its signal ~42x —
letting rare, poorly-evidenced places escape ridge's penalty. Left at 0/1 the
penalty shrinks them, which is what we want.

Excluded as leakage: `price_per_sqft` and `rent_vs_zip_zori` are computed *from*
price; `daysOnMarket` is downstream of it. `yearBuilt` is 85% missing.

In [ ]:
NUMERIC = [
    "bedrooms", "bathrooms", "squareFootage",
    "dist_coast_mi", "dist_ucsc_mi", "dist_csumb_mi",
    "dist_hwy17_mi", "dist_town_center_mi", "dist_sc_downtown_mi",
    "latitude", "longitude",
    "months_since_2020", "listed_month",
    "zori_zip", "tract_median_hh_income", "tract_median_gross_rent",
    "tract_pct_renter", "tract_pop_density_sqmi",
    "sqft_missing", "zori_missing",
]
CATEGORICAL = ["place", "propertyType", "place_kind", "county"]
TARGET = "log_price"

y_train, y_test = train[TARGET], test[TARGET]

def build_ridge(numeric, categorical):
    fill = train[numeric].median()
    enc  = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train[categorical])
    names = enc.get_feature_names_out(categorical)
    sc   = StandardScaler().fit(train[numeric].fillna(fill))
    def build(d):
        num = pd.DataFrame(sc.transform(d[numeric].fillna(fill)), columns=numeric, index=d.index)
        cat = pd.DataFrame(enc.transform(d[categorical]), columns=names, index=d.index)
        return pd.concat([num, cat], axis=1)
    return build(train), build(test)

X_tr, X_te = build_ridge(NUMERIC, CATEGORICAL)
print(f"X_train {X_tr.shape}   X_test {X_te.shape}")

ridge = Ridge(alpha=1.0).fit(X_tr, y_train)
evaluate(np.exp(ridge.predict(X_te)), "ridge (all features)");

### 5b. Ridge for interpretation

The coefficients above are unreadable: `dist_ucsc_mi`, `dist_hwy17_mi` and
`dist_sc_downtown_mi` correlate at **1.00** with each other (UCSC, the Hwy 17
ramps and downtown Santa Cruz are all within a couple of miles), and the 40
`place` dummies reconstruct `dist_ucsc_mi` with R²=0.92. Ridge splits one real
effect across many identical columns and the signs come out backwards.

Dropping the location categoricals and the redundant distances costs 0.4 MAPE
points and buys coefficients that point the right way.

In [ ]:
GEO_NUM = [c for c in NUMERIC
           if c not in ("latitude", "longitude", "dist_hwy17_mi",
                        "dist_sc_downtown_mi", "dist_csumb_mi")]
GEO_CAT = ["propertyType", "place_kind"]      # place and county both encode position

Xg_tr, Xg_te = build_ridge(GEO_NUM, GEO_CAT)
ridge_geo = Ridge(alpha=1.0).fit(Xg_tr, y_train)
evaluate(np.exp(ridge_geo.predict(Xg_te)), "ridge (interpretable)")

coefs = pd.Series(ridge_geo.coef_, index=Xg_tr.columns)[GEO_NUM].sort_values()
print("\neffect on rent per 1 standard deviation:")
for name, v in coefs.items():
    print(f"   {name:<26}{v:>+8.4f}   ~{(np.exp(v)-1)*100:>+6.1f}%")

sd = train[GEO_NUM].std()
print("\nper real unit:")
for k_, unit, mult in [("dist_coast_mi", "per mile inland", 1),
                       ("squareFootage", "per 100 sqft", 100)]:
    print(f"   {unit:<22}{(np.exp(coefs[k_]/sd[k_]*mult)-1)*100:>+7.2f}%")

`sqft_missing` at ~-7% is the largest single effect: a listing that simply
does not state its square footage rents ~7% below an otherwise identical one
that does. Probably proxying for informal landlords and lower-end stock.

`dist_ucsc_mi` is ~-0.1% per mile — essentially nothing. The student-housing
effect we expected is not visible here, likely because it lives in the per-room
market this data source does not cover.

## 6. LightGBM

No imputation, no one-hot, no scaling. Trees split on thresholds so scale is
irrelevant, `NaN` gets its own branch, and categoricals are handled natively.
Collinear features are harmless — trees pick one and ignore the rest — so the
dropped columns come back.

In [ ]:
LGB_NUM = NUMERIC
LGB_CAT = CATEGORICAL
FEATURES = LGB_NUM + LGB_CAT
LGB_PARAMS = dict(n_estimators=600, learning_rate=0.05, num_leaves=31,
                  min_child_samples=20, random_state=42, verbose=-1)

def build_lgb(d, categories=None):
    X = d[FEATURES].copy()
    for c in LGB_CAT:
        X[c] = X[c].astype("category")
        if categories is not None:
            X[c] = X[c].cat.set_categories(categories[c])   # train defines the vocabulary
    return X

X_train_lgb = build_lgb(train)
cats = {c: X_train_lgb[c].cat.categories for c in LGB_CAT}
X_test_lgb  = build_lgb(test, cats)

lgbm = lgb.LGBMRegressor(**LGB_PARAMS).fit(X_train_lgb, y_train)
evaluate(np.exp(lgbm.predict(X_test_lgb)), "lightgbm (nominal)");

## 7. Trees cannot extrapolate

`months_since_2020` runs 1.6–72.0 in train and 72.0–80.7 in test, so **100% of
test rows sit beyond anything the trees saw**. Past the largest split it learned,
every future date lands in the same leaf and returns the same number — the model
predicts 2025 prices for 2027 and would age badly.

**Fix: split the job.** A straight line handles the date; the trees learn only
how far each unit sits from that line ("this one is +22% vs typical"), which does
not depend on the calendar. The line extrapolates indefinitely.

In [ ]:
trend = LinearRegression().fit(train[["months_since_2020"]], train[TARGET])
resid_train = train[TARGET] - trend.predict(train[["months_since_2020"]])

lgbm_resid = lgb.LGBMRegressor(**LGB_PARAMS).fit(X_train_lgb, resid_train)

pred_log = trend.predict(test[["months_since_2020"]]) + lgbm_resid.predict(X_test_lgb)
pred = evaluate(np.exp(pred_log), "LightGBM + trend  <-- MODEL")

print(f"\ntrend: {(np.exp(trend.coef_[0]*12)-1)*100:+.1f}% per year")
print(f"time explains {trend.score(train[['months_since_2020']], train[TARGET]):.1%} "
      f"of rent variation; the other {1-trend.score(train[['months_since_2020']], train[TARGET]):.1%} "
      f"is differences between units")

In [ ]:
# the frozen-prediction problem, before and after
months = np.linspace(40, 100, 120)
grid = pd.concat([X_test_lgb.iloc[[0]]] * len(months), ignore_index=True)
grid["months_since_2020"] = months
for c in LGB_CAT:
    grid[c] = grid[c].astype("category").cat.set_categories(cats[c])

plt.figure(figsize=(8, 3.5))
plt.plot(months, np.exp(lgbm.predict(grid)), label="nominal — freezes")
plt.plot(months, np.exp(trend.predict(pd.DataFrame({"months_since_2020": months}))
                        + lgbm_resid.predict(grid)), label="+ trend — keeps going")
plt.axvline(train.months_since_2020.max(), ls="--", c="grey", label="end of training data")
plt.xlabel("months_since_2020"); plt.ylabel("predicted rent"); plt.legend(); plt.show()

### What did not work: deflating by ZORI

Restating rents in 2026-01 dollars using Zillow's index should remove the trend
without needing the line. It made things worse (MAPE 13.9%, bias +$74) because
**ZORI tracks all unit types**, which rose 3.8% through 2026, while studios and
1BRs were flat (-0.1%). The conversion added growth that never happened in this
segment.

Over six years both grow at +4.3%/yr — the index is fine for a long-run trend and
wrong for month-to-month correction. Kept here as a documented negative result.

## 8. Calibrated prediction intervals

The model is off by ~12.6% typically, so a single number implies precision it
does not have. Worse, 31 one-bedrooms in Pacific Grove are identical on every
field we have and rent from $1,600 to $6,500 — the model cannot see views,
renovations or furnishing, so it predicts the middle and misses both ends.

Quantile regression alone produced intervals that were **too narrow** (54%
coverage against an 80% target) because it learned its sense of spread from
training data it had already fit tightly.

**Conformal calibration** measures the spread instead: hold out the most recent
20% of train, see how wrong the model actually is on listings it never saw, and
use that. Errors live in log space so the half-width is multiplicative.

In [ ]:
cut   = train["months_since_2020"].quantile(0.80)
fit_d = train[train["months_since_2020"] <  cut]
cal_d = train[train["months_since_2020"] >= cut]

trend_tmp = LinearRegression().fit(fit_d[["months_since_2020"]], fit_d[TARGET])
tmp = lgb.LGBMRegressor(**LGB_PARAMS).fit(
    build_lgb(fit_d, cats), fit_d[TARGET] - trend_tmp.predict(fit_d[["months_since_2020"]]))

cal_pred = trend_tmp.predict(cal_d[["months_since_2020"]]) + tmp.predict(build_lgb(cal_d, cats))
k = np.quantile(np.abs(cal_d[TARGET] - cal_pred), 0.80)

# apply k to the full model, which trained on all of train
test["mid"] = np.exp(pred_log)
test["lo"]  = np.exp(pred_log - k)
test["hi"]  = np.exp(pred_log + k)
test["inside"] = test.price.between(test.lo, test.hi)

print(f"calibrated on {len(cal_d)} held-out listings")
print(f"k = {k:.3f} in log space  ->  x/÷ {np.exp(k):.2f}")
print(f"coverage {test.inside.mean():.1%}  (target 80%)   median width ${np.median(test.hi-test.lo):,.0f}\n")
for _, r in test.sample(5, random_state=3).iterrows():
    print(f"   {r.formattedAddress[:38]:<40} actual ${r.price:>6,.0f}   "
          f"${r.lo:>6,.0f} - ${r.hi:<6,.0f}  {'inside' if r.inside else 'OUTSIDE'}")

## 9. Where the model fails

A single MAPE hides a lot. Error is broken out below by sub-market and by rent
level, with `n` alongside — thin neighbourhoods look erratic for that reason
rather than because the market is strange there.

In [ ]:
test["err"] = test["mid"] - test["price"]
test["ape"] = (test["err"] / test["price"]).abs() * 100

by_place = (test.groupby("place")
    .agg(n=("price", "size"), median=("price", "median"),
         MAE=("err", lambda e: e.abs().mean()), MAPE=("ape", "mean"), bias=("err", "mean"))
    .query("n >= 15").sort_values("MAPE"))
print(by_place.round(0).to_string())

test["band"] = pd.qcut(test["price"], 5)
print("\n", test.groupby("band", observed=True)
      .agg(n=("price", "size"), MAPE=("ape", "mean"),
           bias=("err", "mean"), coverage=("inside", "mean")).round(2).to_string())

Two patterns.

**By place**, 9% in Monterey against 17% in the unincorporated areas — best
where there is most data. "Unincorporated" is not one market; it is every parcel
outside a city boundary, from Pebble Beach to the mountains.

**By rent level**, bias runs +$226 → +$93 → +$27 → −$69 → −$508 across the
quintiles. The model **over-predicts cheap units and under-predicts expensive
ones** — regression to the mean, caused by features that cannot see quality plus
a minimum of 20 listings per leaf. Practically: trust it most in the
$1,858–$2,700 band, and expect high-end listings to look "overpriced".

## 10. Ablation

`zori_zip` and the `tract_*` columns are market averages partly derived from the
same rents being predicted. They are legitimately available before a listing
appears, but a model leaning on them would be learning "rents are high here"
rather than "this apartment is nice".

In [ ]:
ZORI  = ["zori_zip", "zori_missing"]
TRACT = ["tract_median_hh_income", "tract_median_gross_rent",
         "tract_pct_renter", "tract_pop_density_sqmi"]

def ablate(drop_num=(), drop_cat=(), label=""):
    num = [c for c in LGB_NUM if c not in drop_num]
    cat = [c for c in LGB_CAT if c not in drop_cat]
    F = num + cat
    def prep(d, c=None):
        X = d[F].copy()
        for col in cat:
            X[col] = X[col].astype("category")
            if c is not None: X[col] = X[col].cat.set_categories(c[col])
        return X
    Xa = prep(train); ca = {c: Xa[c].cat.categories for c in cat}
    p = np.exp(trend.predict(test[["months_since_2020"]])
               + lgb.LGBMRegressor(**LGB_PARAMS).fit(Xa, resid_train).predict(prep(test, ca)))
    e = p - test["price"]
    print(f"   {label:<38}{len(F):>4}{e.abs().mean():>9,.0f}"
          f"{(e.abs()/test['price']).mean()*100:>8.1f}%")

print(f"   {'model':<38}{'feat':>4}{'MAE':>9}{'MAPE':>9}")
ablate(label="FULL")
ablate(ZORI, label="without zori_zip")
ablate(TRACT, label="without tract_*")
ablate(ZORI + TRACT, label="without BOTH market averages")
print()
ablate(("squareFootage", "sqft_missing"), label="without squareFootage")
ablate(("months_since_2020",), label="without months_since_2020")
ablate(drop_cat=["place"], label="without place")
ablate(("dist_coast_mi",), label="without dist_coast_mi")

Dropping **all five market averages costs 0.3 MAPE points** — the model is not
leaning on circular aggregates.

`squareFootage` is the most important feature (+1.9pt when removed), which is
worth noting because it was completely broken until the 13,032 sqft
building-footprint values were fixed in `clean.py`. Note also that split-count
importance ranked `months_since_2020` first; ablation reverses them. How often a
model splits on a feature is not how much it needs it.

`dist_coast_mi` looks worthless at +0.1pt only because `latitude`, `longitude`
and `place` can reconstruct it. One-at-a-time ablation measures whether a feature
is *irreplaceable*, not whether it is useful.

## 11. Summary

In [ ]:
summary = pd.DataFrame(results).drop_duplicates("model", keep="last")
summary["MAE"] = summary["MAE"].map("${:,.0f}".format)
summary["MAPE"] = summary["MAPE"].map("{:.1f}%".format)
summary["bias"] = summary["bias"].map("${:+,.0f}".format)
print(summary.to_string(index=False))
print(f"\ninterval: x/÷ {np.exp(k):.2f}, {test.inside.mean():.0%} coverage on {len(test)} test listings")

### Known limitations

- **No amenity data.** RentCast returns no listing text, so utilities, furnished,
  parking, laundry, pets, views and renovations are all invisible. This is the
  main cause of the regression to the mean in section 9.
- **ADU / cottage coverage.** The feed draws on MLS and syndicated sources, which
  under-represent the detached cottages and in-law units that make up much of the
  real studio/1BR market here. The model describes *professionally listed* units.
- **Asking rent, not contract rent.** Long-tenured below-market tenancies never
  appear.
- **Fixed interval width.** `k` is one number, so every interval is the same
  proportional width. Ideally an unincorporated listing would get a visibly wider
  range than a Monterey one — a per-sub-market `k` is the natural extension.
- **Market level is an assumption.** The model prices a unit *against* the
  prevailing market; where that market sits on a future date comes from a fitted
  line, not from evidence.